# Generate Users Data

Generate random users and write SQL INSERT queries to `user.insert.sql` for seeding.

In [9]:
import sys
print(sys.executable)

d:\HCMUS\Third Year\Ultra Web Skills\ReflourishedOnlineAuction\Online-Auction\data\.venv\Scripts\python.exe


In [10]:
import os
import random
import bcrypt
from datetime import datetime, timedelta
from faker import Faker

fake = Faker('vi_VN')
SALT_ROUNDS = 10
print("✅ Libraries imported")

✅ Libraries imported


In [11]:
emails_seen = set()
usernames_seen = set()

def hash_password(password: str) -> str:
    salt = bcrypt.gensalt(rounds=SALT_ROUNDS)
    hashed = bcrypt.hashpw(password.encode('utf-8'), salt)
    return hashed.decode('utf-8')

def generate_random_user():
    full_name = fake.name()
    while True:
        email = fake.email()
        username = email.split('@')[0]
        if email not in emails_seen and username not in usernames_seen:
            emails_seen.add(email)
            usernames_seen.add(username)
            break
    password = "Lok@1412"
    address = fake.address().replace('\n', ', ')
    return {
        'username': username,
        'full_name': full_name,
        'email': email,
        'password': password,
        'address': address
    }

def generate_random_date_of_birth():
    today = datetime.now()
    start_date = today - timedelta(days=70*365)
    end_date = today - timedelta(days=18*365)
    random_days = random.randint(0, (end_date - start_date).days)
    return (start_date + timedelta(days=random_days)).date()

In [12]:
roles = ['admin'] + ['seller'] * 10 + ['user'] * 19
NUM_USERS = len(roles)
sql_statements = []
sql_statements.append("-- Reset users table")
sql_statements.append("TRUNCATE TABLE users RESTART IDENTITY CASCADE;\n")

print(f"Generating SQL queries for {NUM_USERS} users...")

default_hashed_pwd = hash_password("Lok@1412")

for i in range(1, NUM_USERS + 1):
    user_data = generate_random_user()
    dob = generate_random_date_of_birth()
    role = roles[i - 1]
    
    username = user_data['username'].replace("'", "''")
    full_name = user_data['full_name'].replace("'", "''")
    email = user_data['email'].replace("'", "''")
    address = user_data['address'].replace("'", "''")
    
    sql = f"INSERT INTO users (user_id, username, full_name, email, password, address, role, date_of_birth, status) VALUES ({i}, '{username}', '{full_name}', '{email}', '{default_hashed_pwd}', '{address}', '{role}', '{dob.isoformat()}', 'active');"
    sql_statements.append(sql)

sql_statements.append(f"\n-- Adjust ID sequence")
sql_statements.append(f"SELECT setval('users_user_id_seq', (SELECT MAX(user_id) FROM users));")

output_file = "user.insert.sql"
with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(sql_statements))

print(f"🎉 Successfully generated {output_file} with {NUM_USERS} users!")

Generating SQL queries for 30 users...
🎉 Successfully generated user.insert.sql with 30 users!
